In [1]:
import pydicom
import os
import glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [6]:
df = pd.read_csv("/home/trevor/trevor/repos/3D_SR_CT/test_notebook/ct_scan_metadata.csv")

In [7]:
df

,path,xy_resolution,z_step_size
0,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3996...,0.837891,5.0
1,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3351...,0.933594,5.0
2,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3438...,0.828857,3.0
3,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3291...,0.851562,1.0
4,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3058...,0.494067,3.0
...,...,...,...
1024,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3144...,0.867188,2.0
1025,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3025...,2.000000,0.6
1026,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/4785...,0.837891,2.0
1027,/sdata1/trevor/datasets/Abd_and_Pelvic_CT/3293...,1.032819,3.0


In [8]:
# count studies for which xy_resolution is < 0.75 and z_step_size is < 1.0
print(df[(df['xy_resolution'] <= 1.0) & (df['z_step_size'] <= 2.0)].shape[0])


272


In [9]:
# 0.75 by 2 mm
# count studies for which xy_resolution is < 0.75 and z_step_size is < 2.0
print(df[(df['xy_resolution'] <= 0.75) & (df['z_step_size'] <= 2.0)].shape[0])

84


In [10]:
# 1 mm isotropic
# count studies for which xy_resolution is < 1.0 and z_step_size is < 1.0
print(df[(df['xy_resolution'] <= 1.0) & (df['z_step_size'] <= 1.0)].shape[0])

70


In [11]:
matching_paths = df[(df['xy_resolution'] <= 1.0) & (df['z_step_size'] <= 1.0)]['path'].tolist()
print(matching_paths)

['/sdata1/trevor/datasets/Abd_and_Pelvic_CT/32918612/DICOM/00002543/AA048832/AA4ED959/0000EE94', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/35939053/DICOM/0000FD92/AA5EE6DB/AA372C96/000029D0', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/49043321/DICOM/00003F48/AAFCA411/AAB5ED7A/000027C6', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/35939053/DICOM/0000FD92/AA5EE6DB/AA372C96/00002C49', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/44069539/DICOM/0000FC3B/AAA45B2D/AA1F2E51/000031E7', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/47068179/DICOM/000067E0/AAF5B6C1/AAAD1C46/00009503', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/32609963/DICOM/000004E5/AAB22642/AA827BB3/00000A24', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/32717454/DICOM/00009115/AADB92CB/AA445077/00000D2E', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/31261852/DICOM/00004372/AADFA07A/AA1B6DB5/0000399D', '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/33441286/DICOM/0000FEE6/AAE2C63D/AAB593C2/0000EB6F', '/sdata1/trevor/datasets/Abd_and_Pelvic

In [12]:
# write matching paths to file
with open('matching_paths.txt', 'w') as f:
    for path in matching_paths:
        write_path = str(path).replace('/sdata1/trevor/datasets/', '')
        f.write(write_path + '\n')


In [13]:
# Get unique patient IDs from matching_paths
patient_ids = set()
for path in matching_paths:
    # Extract patient ID (the directory right after Abd_and_Pelvic_CT)
    path_parts = path.split('/')
    # Find the index of 'Abd_and_Pelvic_CT'
    try:
        idx = path_parts.index('Abd_and_Pelvic_CT')
        if idx + 1 < len(path_parts):
            patient_id = path_parts[idx + 1]
            patient_ids.add(patient_id)
    except ValueError:
        continue

print(f"Number of unique patient IDs: {len(patient_ids)}")
print(f"Unique patient IDs: {sorted(patient_ids)}")

Number of unique patient IDs: 59
Unique patient IDs: ['15589556', '16030930', '27046674', '29448312', '31261852', '31870877', '32137442', '32531452', '32609963', '32611670', '32619333', '32717454', '32888375', '32918612', '33441286', '33569633', '33691743', '34058160', '34077053', '34414447', '34500030', '34807143', '34932167', '34982060', '35120184', '35616645', '35774025', '35939053', '36013694', '36741817', '38567820', '38813204', '38954811', '39343137', '39681390', '40882781', '41206528', '41637130', '41928852', '42205059', '42890751', '43060759', '43330811', '44054337', '44069539', '44707239', '45262362', '45361937', '45412528', '46587354', '46645522', '46871626', '47068179', '47502646', '47762476', '48054975', '49043321', '49584236', '50088554']


In [14]:
# write matching paths to file
with open('matching_patient_ids.txt', 'w') as f:
    for path in sorted(patient_ids):
        write_path = str(path).replace('/sdata1/trevor/datasets/', '')
        f.write(write_path + '\n')


In [6]:
image_path = '/sdata1/trevor/datasets/Abd_and_Pelvic_CT/33441286/DICOM/0000FEE6/AAE2C63D/AAB593C2/0000EB6F/EE01BFE5'
pydicom.dcmread(image_path)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 168
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.12.2.1107.5.1.4.55479.30000020040105415612500000618
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.752.24.3.3.25.7
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPI']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.12.2.1107.5.1.4.55479.30000020040105415612500000618
(0008,0020) Study Date                          DA: '20200401'
(0008,0021) Series Date                         